# 02 — Contribution, league factors, replacement level (Phase 2)

Plan: `docs/superpowers/plans/2026-08-28-phase2-contribution.md`. Each step below runs the smallest
thing that produces a real output, the decision is read off it and recorded in the note cell, and
only then the code is ported to `scout.models`. Inputs are the Phase 1 tables (`scout.panel`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 80); pd.set_option("display.max_rows", 300)
from scout import config
from scout.data import understat
from scout.panel import market, player_match, stints, workrate
LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}

## Step 1 — Which per-90 quantities carry signal, per role, and at what minutes floor?

Candidates from `player_match` (npxG, xA, key passes, shots, xGChain, xGBuildup) plus goals and
assists as the raw baseline. Per player × league-season × role (a player's season in one role),
per 90. Criterion: year-to-year r ≥ 0.3 for the same player in the same role, at floors 300 / 600 /
900 / 1,200 minutes; the floor is the lowest one at which r stops rising materially, weighed against
the share of the backtest population (paid Big-5 departures with a panel row) it drops.

In [2]:
pm = player_match.build()
pm["competition_id"] = pm.league.map(LEAGUE_TO_COMP)
shots = understat.load("shots")
print("shots columns:", shots.columns.tolist())
# penalties: situation NA with xG 0.7612 (notebook 01, Part 4b); own goals are not the shooter's
is_pen = shots.situation.isna() & (shots.xg.round(4) == 0.7612)
pen_xg = shots[is_pen].groupby(["game_id", "player_id"]).xg.sum().rename("pen_xg")
pm = pm.merge(pen_xg, on=["game_id", "player_id"], how="left").fillna({"pen_xg": 0.0})
pm["npxg"] = pm.xg - pm.pen_xg
print(len(pm), "player-match rows | penalty xG subtracted from", int((pm.pen_xg > 0).sum()), "rows | own-goal rows in shots:", int((shots.result == "OwnGoal").sum()) if "result" in shots else "n/a")

QUANTITIES = ["npxg", "xa", "key_passes", "shots", "xg_chain", "xg_buildup", "goals", "assists"]
season_role = pm.dropna(subset=["role"]).groupby(["competition_id", "season", "player_id", "role"]).agg(minutes=("minutes", "sum"), **{q: (q, "sum") for q in QUANTITIES}).reset_index()
per90 = season_role.copy()
for q in QUANTITIES:
    per90[q] = per90[q] / per90.minutes * 90
print(len(per90), "player-season-roles |", per90.groupby("role").size().to_dict())

shots columns: ['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']
630291 player-match rows | penalty xG subtracted from 1092 rows | own-goal rows in shots: 0
42157 player-season-roles | {'CB': 6770, 'CM': 8472, 'FB': 7317, 'GK': 2376, 'ST': 6179, 'W': 11043}


In [3]:
FLOORS = [300, 600, 900, 1200]


def year_to_year(frame, floor, quantities):
    kept = frame[frame.minutes >= floor]
    nxt = kept.assign(season=kept.season - 1)
    pairs = kept.merge(nxt, on=["competition_id", "player_id", "role", "season"], suffixes=("", "_next"))
    return pd.Series({q: pairs[q].corr(pairs[f"{q}_next"]) for q in quantities}), len(pairs)


rows = []
for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    for floor in FLOORS:
        r, n = year_to_year(per90[per90.role == role], floor, QUANTITIES)
        rows.append(pd.concat([pd.Series({"role": role, "floor": floor, "pairs": n}), r.round(2)]))
stability = pd.DataFrame(rows).set_index(["role", "floor"])
print(stability.to_string())
print("\nquantities with r >= 0.3 at 900, per role:")
for role, group in stability.xs(900, level="floor").iterrows():
    print(f"  {role}: {[q for q in QUANTITIES if group[q] >= 0.3]}")

            pairs  npxg    xa  key_passes  shots  xg_chain  xg_buildup  goals  assists
role floor                                                                            
GK   300     1020  0.06  0.04        0.12   0.02      0.56        0.56  -0.00     0.01
     600      890  0.15  0.07        0.11   0.05      0.65        0.65  -0.00     0.04
     900      800  0.15  0.06        0.12   0.08      0.64        0.64  -0.00     0.07
     1200     733  0.16  0.07        0.16   0.13      0.64        0.64  -0.00     0.08
CB   300     2966  0.27  0.21        0.40   0.46      0.67        0.67   0.13     0.06
     600     2492  0.30  0.24        0.45   0.52      0.71        0.71   0.16     0.08
     900     2047  0.33  0.28        0.49   0.55      0.73        0.72   0.15     0.10
     1200    1675  0.38  0.26        0.44   0.57      0.75        0.74   0.15     0.11
FB   300     2686  0.45  0.48        0.61   0.61      0.62        0.61   0.24     0.23
     600     2121  0.52  0.53        0.65  

In [4]:
# The cost side of the floor: paid departures from Big-5 clubs (backtest population) kept at each floor
moves = market.build()
paid = moves[(moves.kind == "paid")].copy()
paid["transfer_season"] = ("20" + paid.transfer_season.str[:2]).astype(int)
paid["prev_season"] = paid.transfer_season - (~pd.to_datetime(paid.transfer_date).dt.month.isin([1, 2, 3])).astype(int)
big5_clubs = stints.tm.load_player_club_seasons(list(config.BIG5), list(config.SEASONS))[["club_id"]].drop_duplicates()
paid = paid[paid.from_club_id.isin(big5_clubs.club_id) & paid.transfer_season.between(2015, 2024)]
last = stints.build(list(config.BIG5) + list(config.FEEDERS), list(config.SEASONS))[["tm_player_id", "club_id", "season", "minutes"]]
present = paid.merge(last, left_on=["player_id", "from_club_id", "prev_season"], right_on=["tm_player_id", "club_id", "season"]).dropna(subset=["minutes"])
print(f"paid Big-5 departures 15/16 → 24/25 with a panel row at the selling club: {len(present)} (of {len(paid)})")
print("kept at each floor:", {f: f"{(present.minutes >= f).mean():.1%}" for f in FLOORS}, "| fee share kept:", {f: f"{present.transfer_fee[present.minutes >= f].sum() / present.transfer_fee.sum():.1%}" for f in FLOORS})

paid Big-5 departures 15/16 → 24/25 with a panel row at the selling club: 1706 (of 2736)
kept at each floor: {300: '85.2%', 600: '77.6%', 900: '69.6%', 1200: '60.7%'} | fee share kept: {300: '93.9%', 600: '89.4%', 900: '84.5%', 1200: '76.5%'}


Work-rate quantities (Sofascore per 90, the 16 shared metrics of Step 5d plus the keeper block)
need the identity join: Sofascore id → Transfermarkt id ← Understat id, then the Understat role.

In [5]:
from scout.data import reep, sofascore, transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import identity

comps = list(config.BIG5) + list(config.FEEDERS)
tm_panel = tm_loader.load_player_club_seasons(comps, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
ss = sofascore.load()
us = understat.load("player_season"); us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(tm_clubs, {
    "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(),
    "understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"}),
}, load_overrides("teams"))
people = reep.load_people(); tm_side = identity.transfermarkt_side(tm_panel)
ss_ids = identity.resolve_provider("sofascore", ss, tm_side, lineage, people).drop_duplicates("provider_id")[["provider_id", "tm_player_id"]]
us_ids = identity.resolve_provider("understat", us, tm_side, lineage, people).drop_duplicates("provider_id")[["provider_id", "tm_player_id"]]
print("ids:", len(ss_ids), "sofascore |", len(us_ids), "understat")

wr = pd.concat([ss[["competition_id", "season", "sofascore_player_id", "minutesPlayed"]], workrate.sofascore_per90(ss)], axis=1)
wr["tm_player_id"] = wr.sofascore_player_id.astype(int).astype(str).map(ss_ids.set_index("provider_id").tm_player_id)
roles = per90[["competition_id", "season", "player_id", "role", "minutes"]].copy()
roles["tm_player_id"] = roles.player_id.astype(int).astype(str).map(us_ids.set_index("provider_id").tm_player_id)
main_role = roles.sort_values("minutes", ascending=False).drop_duplicates(["competition_id", "season", "tm_player_id"])
wr = wr.dropna(subset=["tm_player_id"]).merge(main_role[["competition_id", "season", "tm_player_id", "role"]], on=["competition_id", "season", "tm_player_id"])
wr["minutes"] = pd.to_numeric(wr.minutesPlayed); wr["player_id"] = wr.tm_player_id
WR = [m for m in workrate.SHARED if m not in ("xg", "xa", "goals", "assists")] + ["possession_won_att_third_sofascore"]
print(len(wr), "Sofascore player-seasons with a Big-5 role")
rows = []
for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    for floor in [300, 600, 900]:
        r, n = year_to_year(wr[wr.role == role], floor, WR)
        rows.append(pd.concat([pd.Series({"role": role, "floor": floor, "pairs": n}), r.round(2)]))
wr_stability = pd.DataFrame(rows).set_index(["role", "floor"])
print(wr_stability.rename(columns={"possession_won_att_third_sofascore": "poss_won_att3"}).to_string())
print("\nwork-rate quantities with r >= 0.3 at 600, per role:")
for role, group in wr_stability.xs(600, level="floor").iterrows():
    print(f"  {role}: {[q for q in WR if group[q] >= 0.3]}")

ids: 21090 sofascore | 9531 understat
26557 Sofascore player-seasons with a Big-5 role
            pairs  tackles  interceptions  recoveries  clearances  dribbles  key_passes  big_chances_created  accurate_passes  accurate_long_balls  fouls  saves  goals_conceded  poss_won_att3
role floor                                                                                                                                                                                     
GK   300      920     0.07           0.17        0.39        0.45      0.23        0.13                 0.06             0.71                 0.70   0.07   0.30            0.40          -0.01
     600      807     0.07           0.22        0.53        0.51      0.27        0.13                 0.09             0.75                 0.71   0.08   0.36            0.47          -0.01
     900      729     0.08           0.25        0.56        0.49      0.36        0.14                 0.06             0.78                 0.7

/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/mihailandreev/foot

### Step 1 note — quantities per role and the minutes floor, from the two tables above

**Floor: 600 minutes** in a role-season. On the Understat quantities, going from 300 to 600 buys
+0.05 to +0.09 in year-to-year r for every outfield role (ST npxG 0.54 → 0.59, W xA 0.44 → 0.50,
CB shots 0.46 → 0.52); 600 to 900 buys only +0.03 to +0.05 while dropping eight more points of the
backtest population (paid Big-5 departures with a panel row: 77.6% kept at 600 and 89.4% of the
fees, versus 69.6% / 84.5% at 900, 85.2% / 93.9% at 300). Rejected: 300 (the noisiest step and the
largest gain forgone), 900 and 1,200 (less backtest for a smaller gain). Players between 300 and
600 minutes stay in the tables and get profiled from other seasons; they carry no role-season
of their own.

**Quantities that enter, per role** (year-to-year r ≥ 0.3 at 600; goals and assists are never
inputs — the spec builds on expected quantities, and they are also the least stable column in
every role, 0.15–0.45):

| Role | Understat | Work-rate (Sofascore/FotMob shared per 90) |
|---|---|---|
| GK | xGChain, xGBuildup (distribution only, 0.65) | recoveries, clearances, accurate passes, long balls, saves (0.36), goals conceded (0.47 — a team quantity; the keeper's own measure is the Step 6 proxy) |
| CB | npxG (0.30), key passes, shots, xGChain, xGBuildup — not xA (0.24) | tackles, interceptions, recoveries, clearances, dribbles, key passes, accurate passes, long balls, fouls, possession won in the attacking third — not big chances created (0.17) |
| FB, CM, W, ST | npxG, xA, key passes, shots, xGChain, xGBuildup | all twelve, big chances created included (0.36–0.53) |

Two things to carry forward: `goals_conceded` per 90 is stable for outfielders (0.34–0.43) because
it is the team's defence, not the player's — it belongs to plus-minus (Step 2/6), not to a
player's quantity list; and the most stable columns everywhere are volume/style measures
(accurate passes 0.8+, dribbles 0.7+), which say what a player *does*, not how much it is worth —
Step 2 decides what contribution is made of, Step 1 only says which columns carry signal.

Ported: `scout.models.quantities` (`MIN_MINUTES`, `ROLE_QUANTITIES`, `season_role_per90`).

### Step 1 check — `scout.models.quantities` reproduces the cells above

In [6]:
from scout.models import quantities

packaged = quantities.season_role_per90(pm.drop(columns=["pen_xg", "npxg"]), shots)
print(len(packaged), "player-season-roles (cell above: 42,157) | floor:", quantities.MIN_MINUTES)
r, n = year_to_year(packaged[packaged.role == "ST"], 600, ["npxg", "xa", "key_passes", "shots", "xg_chain", "xg_buildup"])
print("ST at 600 — pairs:", n, "(above: 1,408) |", r.round(2).to_dict())
print("(above: npxg 0.59, xa 0.45, key_passes 0.63, shots 0.63, xg_chain 0.60, xg_buildup 0.57)")

42157 player-season-roles (cell above: 42,157) | floor: 600
ST at 600 — pairs: 1408 (above: 1,408) | {'npxg': 0.59, 'xa': 0.45, 'key_passes': 0.63, 'shots': 0.63, 'xg_chain': 0.6, 'xg_buildup': 0.57}
(above: npxg 0.59, xa 0.45, key_passes 0.63, shots 0.63, xg_chain 0.60, xg_buildup 0.57)


## Step 2 — Contribution variant for attacking roles (the open choice in spec §4.A)

Four candidates per player × league-season × club × role at ≥ 600 minutes, all per 90:
(a) **raw** expected output = npxG + xA; (b) **team-share** = the player's npxG + xA divided by his
team's npxG while he is on the pitch (team match npxG × minutes share — the within-match split is
proportional, the best the data allows without on-pitch xG); (c) **plus-minus** = the team's npxG
difference while he is on the pitch, same proportional split; (d) **both** = (b) and (c) together.
Baseline: raw goals + assists per 90. Kill checks (§5.3): year-to-year stability must beat G+A;
correlation with the team's season xG difference; and the decider — which variant, measured at
club A in season s, best predicts the player's raw expected output at a *different* club in s+1.

In [7]:
from scout.panel import team_season

tm_long = team_season.team_match_long(understat.load("team_match"))
tm_long["competition_id"] = tm_long.league.map(LEAGUE_TO_COMP)
team_game = tm_long[["competition_id", "season", "game_id", "team_id", "np_xg_for", "np_xg_against"]]
rows = pm.merge(team_game, on=["competition_id", "season", "game_id", "team_id"], how="inner")
share = rows.minutes / 90
rows["on_pitch_np_xg_for"] = rows.np_xg_for * share
rows["on_pitch_np_xg_diff"] = (rows.np_xg_for - rows.np_xg_against) * share
KEYS = ["competition_id", "season", "player_id", "team_id", "role"]
stint = rows.dropna(subset=["role"]).groupby(KEYS).agg(minutes=("minutes", "sum"), npxg=("npxg", "sum"), xa=("xa", "sum"), goals=("goals", "sum"), assists=("assists", "sum"),
                                                      team_for=("on_pitch_np_xg_for", "sum"), team_diff=("on_pitch_np_xg_diff", "sum")).reset_index()
stint = stint[stint.minutes >= quantities.MIN_MINUTES].copy()
per = 90 / stint.minutes
stint["raw"] = (stint.npxg + stint.xa) * per
stint["team_share"] = (stint.npxg + stint.xa) / stint.team_for
stint["plus_minus"] = stint.team_diff * per
stint["ga"] = (stint.goals + stint.assists) * per
print(len(stint), "club-season-roles at ≥600 min |", stint.role.value_counts().to_dict())
VARIANTS = ["raw", "team_share", "plus_minus", "ga"]

23086 club-season-roles at ≥600 min | {'CM': 5052, 'W': 4809, 'CB': 4579, 'FB': 4101, 'ST': 2949, 'GK': 1596}


In [8]:
# Kill check 1: year-to-year stability (same player, same role, consecutive seasons — any club) vs G+A
def stability_by_role(frame, cols):
    out = {}
    for role, group in frame.groupby("role"):
        season_level = group.groupby(["competition_id", "season", "player_id", "role"])[cols + ["minutes"]].agg({**{c: "mean" for c in cols}, "minutes": "sum"}).reset_index()
        nxt = season_level.assign(season=season_level.season - 1)
        pairs = season_level.merge(nxt, on=["competition_id", "season", "player_id", "role"], suffixes=("", "_next"))
        out[role] = {c: round(pairs[c].corr(pairs[f"{c}_next"]), 2) for c in cols} | {"pairs": len(pairs)}
    return pd.DataFrame(out).T
print("year-to-year r by role:"); print(stability_by_role(stint, VARIANTS).to_string())
# Kill check 2: correlation with the team's season npxG difference (does the measure track team quality?)
team_diff_season = tm_long.groupby(["competition_id", "season", "team_id"]).apply(lambda g: (g.np_xg_for - g.np_xg_against).mean()).rename("team_season_diff").reset_index()
with_team = stint.merge(team_diff_season, on=["competition_id", "season", "team_id"])
print("\ncorrelation with team season npxG difference, attacking roles (W, ST):")
att = with_team[with_team.role.isin(["W", "ST"])]
print({v: round(att[v].corr(att.team_season_diff), 2) for v in VARIANTS})

year-to-year r by role:
     raw  team_share  plus_minus    ga   pairs
CB  0.32        0.27        0.69  0.15  2481.0
CM  0.69        0.62        0.71  0.50  2656.0
FB  0.60        0.51        0.68  0.37  2109.0
GK  0.08        0.11        0.67  0.06   889.0
ST  0.60        0.34        0.65  0.49  1391.0
W   0.64        0.43        0.68  0.45  2159.0

correlation with team season npxG difference, attacking roles (W, ST):
{'raw': np.float64(0.55), 'team_share': np.float64(0.0), 'plus_minus': np.float64(0.93), 'ga': np.float64(0.48)}


In [9]:
# Kill check 3 (decider): movers — same player, ≥600 min at club A in season s and at a different club B in s+1
nxt = stint.assign(season=stint.season - 1).rename(columns={"team_id": "team_next", "competition_id": "comp_next"})
movers = stint.merge(nxt[["comp_next", "season", "player_id", "role", "team_next", "raw", "ga"]].rename(columns={"raw": "raw_next", "ga": "ga_next"}), on=["season", "player_id", "role"])
movers = movers[movers.team_next != movers.team_id]
stayers = stint.merge(nxt[["comp_next", "season", "player_id", "role", "team_next", "raw"]].rename(columns={"raw": "raw_next"}), on=["season", "player_id", "role"])
stayers = stayers[stayers.team_next == stayers.team_id]
print("movers:", len(movers), "| stayers:", len(stayers), "| movers by role:", movers.role.value_counts().to_dict())


def predict_next(frame, variant, target="raw_next"):
    x, y = frame[variant], frame[target]
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    return round(x.corr(y), 3), round(float(np.sqrt((resid ** 2).mean())), 3)


results = {}
for role_set, label in [(["W", "ST"], "attacking (W, ST)"), (["CB", "FB", "CM"], "non-attacking (CB, FB, CM)")]:
    m = movers[movers.role.isin(role_set)]
    s = stayers[stayers.role.isin(role_set)]
    results[label] = {"n_movers": len(m)}
    for v in VARIANTS:
        results[label][f"{v} r / rmse"] = predict_next(m, v)
    both = m.copy(); both["both"] = np.polyfit(np.c_[m.team_share, m.plus_minus].T.tolist()[0], m.raw_next, 1)[0] * 0  # placeholder replaced below
    X = np.c_[np.ones(len(m)), m.team_share, m.plus_minus]
    beta, *_ = np.linalg.lstsq(X, m.raw_next, rcond=None)
    pred = X @ beta
    results[label]["both (share + plus-minus) r / rmse"] = (round(np.corrcoef(pred, m.raw_next)[0, 1], 3), round(float(np.sqrt(((m.raw_next - pred) ** 2).mean())), 3))
    results[label]["stayers: raw r"] = predict_next(s, "raw")[0]
print(pd.DataFrame(results).to_string())

movers: 3079 | stayers: 9973 | movers by role: {'CM': 679, 'CB': 616, 'W': 565, 'ST': 522, 'FB': 493, 'GK': 204}
                                   attacking (W, ST) non-attacking (CB, FB, CM)
n_movers                                        1087                       1788
raw r / rmse                          (0.447, 0.179)             (0.561, 0.081)
team_share r / rmse                    (0.315, 0.19)             (0.521, 0.084)
plus_minus r / rmse                   (0.259, 0.194)             (0.144, 0.097)
ga r / rmse                           (0.375, 0.186)             (0.485, 0.086)
both (share + plus-minus) r / rmse    (0.416, 0.182)             (0.545, 0.082)
stayers: raw r                                 0.693                      0.714


### Step 2 note — contribution variant, from the three kill checks above

**Chosen: raw expected output per 90 — npxG + xA — with no team adjustment**, for every outfield
role. On 23,086 club-season-roles at ≥ 600 minutes:

| | raw | team-share | plus-minus | both | G+A |
|---|---|---|---|---|---|
| year-to-year r, W / ST | 0.64 / 0.60 | 0.43 / 0.34 | 0.68 / 0.65 | – | 0.45 / 0.49 |
| r with the team's season npxG difference (W, ST) | 0.55 | 0.00 | 0.93 | – | 0.48 |
| predicts output at a different club next season, 1,087 attacking movers (r / rmse) | **0.447 / 0.179** | 0.315 / 0.190 | 0.259 / 0.194 | 0.416 / 0.182 | 0.375 / 0.186 |
| same, 1,788 non-attacking movers | **0.561 / 0.081** | 0.521 / 0.084 | 0.144 / 0.097 | 0.545 / 0.082 | 0.485 / 0.086 |

- Kill check 1 passes for raw: it beats goals + assists in stability for every role (CB 0.32 vs
  0.15 up to CM 0.69 vs 0.50).
- Team-share is *less* stable than raw and predicts worse — dividing by the team's on-pitch
  volume adds the team's noise and strips signal that travels with the player. Rejected.
- Plus-minus (proportional split of the team's on-pitch npxG difference) is the most stable
  column and correlates 0.93 with the team's quality: it measures the team. It does not travel
  (0.26 / 0.14 on movers). Rejected as a contribution variant; **the spec's "plus-minus is
  required for non-attacking roles" does not survive this form** — Step 6 tests a real on/off
  design (team npxG against in matches the player played versus missed, same season) before
  defensive value is claimed from anything.
- "Both" is fit in-sample on the same movers and still loses to raw. Rejected.
- Stayers' raw r is 0.69–0.71: moving club costs about a quarter of the predictability — the
  size of the context effect Phases 3–4 (league factors, fit) have to explain.

Ported: `scout.models.contribution.expected_output` (the core; intervals, recency and the
opponent slope are added by later steps).

## Step 4 — Recency weighting

Candidates for turning a player's history into one number: last season only; the plain mean of
the last three; exponential decay with half-life 1, 1.5, 2 seasons. Test set: every player-role
with ≥ 600 minutes in each of seasons s−3, s−2, s−1 and s (same role, any club); target = raw
expected output in s. Criterion: best next-season prediction (r, rmse).

In [10]:
season_level = per90.copy()
season_level["expected_output"] = season_level.npxg + season_level.xa
season_level = season_level[season_level.minutes >= quantities.MIN_MINUTES][["competition_id", "season", "player_id", "role", "expected_output", "minutes"]]
# one row per player-role-season (a mover across leagues in one season keeps the bigger stint)
season_level = season_level.sort_values("minutes", ascending=False).drop_duplicates(["season", "player_id", "role"])
wide = season_level.pivot(index=["player_id", "role"], columns="season", values="expected_output")
cases = []
for s in range(2017, 2026):
    block = wide[[s - 3, s - 2, s - 1, s]].dropna()
    cases.append(block.set_axis(["lag3", "lag2", "lag1", "target"], axis=1).assign(season=s))
cases = pd.concat(cases).reset_index()
print(len(cases), "player-role cases with four consecutive ≥600-minute seasons |", cases.role.value_counts().to_dict())


def weighted(frame, half_life):
    w = np.array([0.5 ** (k / half_life) for k in (2, 1, 0)])  # lag3, lag2, lag1
    return (frame[["lag3", "lag2", "lag1"]].to_numpy() * w).sum(axis=1) / w.sum()


schemes = {"last season only": cases.lag1, "mean of three": cases[["lag3", "lag2", "lag1"]].mean(axis=1),
           "half-life 1": weighted(cases, 1.0), "half-life 1.5": weighted(cases, 1.5), "half-life 2": weighted(cases, 2.0), "half-life 3": weighted(cases, 3.0)}
table = {}
for name, pred in schemes.items():
    pred = pd.Series(np.asarray(pred), index=cases.index)
    table[name] = {"r": round(pred.corr(cases.target), 3), "rmse": round(float(np.sqrt(((cases.target - pred) ** 2).mean())), 4)}
    for role in ["W", "ST", "CM"]:
        m = cases.role == role
        table[name][f"r {role}"] = round(pred[m].corr(cases.target[m]), 3)
print(pd.DataFrame(table).T.to_string())

5228 player-role cases with four consecutive ≥600-minute seasons | {'CM': 1256, 'CB': 1192, 'FB': 910, 'W': 818, 'ST': 618, 'GK': 434}
                      r    rmse    r W   r ST   r CM
last season only  0.876  0.1198  0.615  0.618  0.694
mean of three     0.897  0.1074  0.670  0.631  0.717
half-life 1       0.898  0.1069  0.670  0.650  0.732
half-life 1.5     0.899  0.1063  0.674  0.648  0.731
half-life 2       0.899  0.1063  0.674  0.645  0.729
half-life 3       0.899  0.1065  0.674  0.642  0.726


## Step 5 — Finishing residual

Goals minus xG per 90 (both include penalties, so the residual is finishing only). Does it
persist year to year, and does last season's residual add anything to next season's goals beyond
xG? Spec §5.3 expects it to fail; that is the finding either way. Kept as its own column.

In [11]:
xg_totals = pm.dropna(subset=["role"]).groupby(["competition_id", "season", "player_id", "role"]).agg(xg_total=("xg", "sum")).reset_index()
fin = per90[per90.minutes >= quantities.MIN_MINUTES].merge(xg_totals, on=["competition_id", "season", "player_id", "role"])
fin["xg_per90"] = fin.xg_total / fin.minutes * 90
fin["residual"] = fin.goals - fin.xg_per90
print("finishing residual by role — mean, sd:"); print(fin.groupby("role").residual.agg(["mean", "std", "size"]).round(3).to_string())
nxt = fin.assign(season=fin.season - 1)[["competition_id", "season", "player_id", "role", "residual", "goals", "xg_per90"]].rename(columns={"residual": "residual_next", "goals": "goals_next", "xg_per90": "xg_next"})
pairs = fin.merge(nxt, on=["competition_id", "season", "player_id", "role"])
print("\nyear-to-year r of the residual:", pairs.groupby("role").apply(lambda g: round(g.residual.corr(g.residual_next), 3)).to_dict())
for role in ["ST", "W", "CM"]:
    g = pairs[pairs.role == role]
    X1 = np.c_[np.ones(len(g)), g.xg_next]
    X2 = np.c_[np.ones(len(g)), g.xg_next, g.residual]
    r1 = np.corrcoef(X1 @ np.linalg.lstsq(X1, g.goals_next, rcond=None)[0], g.goals_next)[0, 1]
    r2 = np.corrcoef(X2 @ np.linalg.lstsq(X2, g.goals_next, rcond=None)[0], g.goals_next)[0, 1]
    print(f"{role}: next-season goals/90 predicted from next-season xG alone r={r1:.3f}; adding last season's residual r={r2:.3f} (n={len(g)})")

finishing residual by role — mean, sd:
       mean    std  size
role                    
CB   -0.008  0.042  4565
CM   -0.004  0.065  5045
FB   -0.005  0.046  4098
GK     -0.0  0.003  1592
ST   -0.027  0.139  2941
W    -0.004  0.109  4829

year-to-year r of the residual: {'CB': 0.026, 'CM': 0.018, 'FB': 0.015, 'GK': 0.06, 'ST': 0.089, 'W': 0.068}
ST: next-season goals/90 predicted from next-season xG alone r=0.799; adding last season's residual r=0.801 (n=1408)
W: next-season goals/90 predicted from next-season xG alone r=0.764; adding last season's residual r=0.766 (n=2184)
CM: next-season goals/90 predicted from next-season xG alone r=0.770; adding last season's residual r=0.770 (n=2670)


### Step 4 note — recency, from the table above

5,228 player-role cases with four consecutive ≥ 600-minute seasons. Last season alone predicts
the next at r 0.876 / rmse 0.120; pooling three seasons lifts every scheme to 0.897–0.899 (within
role: W 0.67, ST 0.65, CM 0.73 — the pooled r is inflated by between-role spread). Half-lives 1
to 3 differ by ≤ 0.001 in r; **half-life 1.5 has the lowest rmse (0.1063)** and is kept. Rejected:
last season only (clearly worse); the plain mean (same r, rmse 0.1074). Ported:
`scout.models.recency` — missing lags renormalise over what exists.

### Step 5 note — finishing residual, from the output above

Goals minus xG per 90 does not persist: year-to-year r is 0.09 for strikers, 0.07 for wingers,
0.02 for the rest; adding last season's residual to next season's xG changes the prediction of
next season's goals by at most +0.002 in r (ST 0.799 → 0.801). The spec's kill check (§5.3)
expected this failure and it is the finding: finishing is not a stable skill at this sample size,
so contribution stays on expected quantities and the residual is stored as its own descriptive
column, never folded in. Side note for the writeup: strikers score 0.027 goals per 90 fewer than
their xG on average (sd 0.14), the others within 0.01 — Understat's model is well calibrated
by role.

### Step 4 check — `scout.models.recency` reproduces the half-life 1.5 row

In [12]:
from scout.models import recency

pred = recency.weighted_history(cases[["lag1", "lag2", "lag3"]])
print("half-life", recency.HALF_LIFE, "| r:", round(pred.corr(cases.target), 3), "| rmse:", round(float(np.sqrt(((cases.target - pred) ** 2).mean())), 4), "(above: 0.899 / 0.1063)")

half-life 1.5 | r: 0.899 | rmse: 0.1063 (above: 0.899 / 0.1063)


## Step 6 — Non-attacking roles and keepers

**Defensive value, a real on/off.** Per player × club × season: the team's non-penalty xG against
per 90 in the matches he played ≥ 45 minutes versus the matches he did not play at all (same club,
same season; at least 5 matches on each side). `on_off_xga` = off − on, positive when the team
concedes less with him. The same for xG for. Kill checks as in Step 2: year-to-year stability and
whether it travels with a mover.

**Keepers.** Three candidates: (i) the Understat proxy — xG of on-target shots faced minus goals
conceded, per 90 (own goals excluded on both sides); (ii) FotMob `_goals_prevented`; (iii)
Sofascore saves per 90 (Step 1: r 0.36). Criterion: year-to-year r ≥ 0.3 (Phase 0 found 0.4–0.5).

In [13]:
# on/off: every (club, season, match) the club played, joined to the player's minutes in it
club_matches = team_game.rename(columns={"team_id": "club_team_id"})
appearances = pm.dropna(subset=["role"])[["competition_id", "season", "game_id", "team_id", "player_id", "role", "minutes"]]
player_clubs = appearances.groupby(["competition_id", "season", "team_id", "player_id"]).agg(role=("role", lambda r: r.value_counts().index[0]), minutes=("minutes", "sum")).reset_index()
player_clubs = player_clubs[player_clubs.minutes >= quantities.MIN_MINUTES]
grid = player_clubs.merge(club_matches, left_on=["competition_id", "season", "team_id"], right_on=["competition_id", "season", "club_team_id"])
grid = grid.merge(appearances[["competition_id", "season", "game_id", "team_id", "player_id", "minutes"]].rename(columns={"minutes": "mins_in_match"}), on=["competition_id", "season", "game_id", "team_id", "player_id"], how="left").fillna({"mins_in_match": 0})
grid["state"] = np.select([grid.mins_in_match >= 45, grid.mins_in_match == 0], ["on", "off"], "partial")
agg = grid[grid.state != "partial"].groupby(["competition_id", "season", "team_id", "player_id", "role", "state"]).agg(matches=("game_id", "size"), xga=("np_xg_against", "mean"), xgf=("np_xg_for", "mean")).unstack("state")
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
onoff = agg[(agg.matches_on >= 5) & (agg.matches_off >= 5)].reset_index()
onoff["on_off_xga"] = onoff.xga_off - onoff.xga_on
onoff["on_off_xgf"] = onoff.xgf_on - onoff.xgf_off
print(len(onoff), "player-club-seasons with ≥5 matches on and off |", onoff.role.value_counts().to_dict())
print("on/off xGA by role — mean, sd:"); print(onoff.groupby("role").on_off_xga.agg(["mean", "std"]).round(3).to_string())

17327 player-club-seasons with ≥5 matches on and off | {'CM': 3788, 'CB': 3758, 'W': 3410, 'FB': 3350, 'ST': 2058, 'GK': 963}
on/off xGA by role — mean, sd:
       mean    std
role              
CB    0.006  0.317
CM    0.002   0.32
FB   -0.002  0.315
GK     0.02  0.301
ST    0.034  0.331
W     0.011  0.334


In [14]:
def onoff_stability(frame, col):
    season_level = frame.sort_values("matches_on", ascending=False).drop_duplicates(["season", "player_id", "role"])
    nxt = season_level.assign(season=season_level.season - 1)
    pairs = season_level.merge(nxt, on=["season", "player_id", "role"], suffixes=("", "_next"))
    stay = pairs[pairs.team_id == pairs.team_id_next]; move = pairs[pairs.team_id != pairs.team_id_next]
    return pd.Series({"stayers r": round(stay[col].corr(stay[f"{col}_next"]), 3), "n stay": len(stay), "movers r": round(move[col].corr(move[f"{col}_next"]), 3), "n move": len(move)})

print("on/off xGA — year-to-year, same club vs after a move:")
print(pd.DataFrame({role: onoff_stability(onoff[onoff.role == role], "on_off_xga") for role in ["GK", "CB", "FB", "CM", "W", "ST"]}).T.to_string())
print("\non/off xG for:")
print(pd.DataFrame({role: onoff_stability(onoff[onoff.role == role], "on_off_xgf") for role in ["CB", "FB", "CM", "W", "ST"]}).T.to_string())
print("\nfor comparison, raw expected output on the same movers/stayers definition: attacking movers r 0.447, stayers 0.69 (Step 2)")

on/off xGA — year-to-year, same club vs after a move:
    stayers r  n stay  movers r  n move
GK      0.019   229.0     0.092    94.0
CB      0.078  1415.0    -0.038   424.0
FB     -0.025  1183.0    -0.038   345.0
CM      0.061  1197.0     0.075   423.0
W      -0.018   812.0     0.044   321.0
ST      0.076   440.0     0.050   283.0

on/off xG for:
    stayers r  n stay  movers r  n move
CB     -0.009  1415.0    -0.050   424.0
FB      0.003  1183.0     0.112   345.0
CM      0.030  1197.0     0.078   423.0
W      -0.004   812.0    -0.052   321.0
ST      0.080   440.0     0.110   283.0

for comparison, raw expected output on the same movers/stayers definition: attacking movers r 0.447, stayers 0.69 (Step 2)


In [15]:
# Keeper proxy from the shots table: on-target xG faced minus goals conceded, per 90 of the keeper's minutes
on_target = shots[shots.result.isin(["Goal", "Saved Shot"])].copy()
on_target["competition_id"] = on_target.league.map(LEAGUE_TO_COMP)
faced = on_target.groupby(["competition_id", "season", "game_id", "team_id"]).agg(xg_on_target=("xg", "sum"), goals=("result", lambda r: (r == "Goal").sum())).reset_index()
keepers = pm[(pm.role == "GK") & (pm.minutes >= 45)][["competition_id", "season", "game_id", "team_id", "player_id", "minutes"]]
# the shots a keeper faces are the *other* team's shots in his game
opponent = team_game[["competition_id", "season", "game_id", "team_id"]].merge(team_game[["competition_id", "season", "game_id", "team_id"]].rename(columns={"team_id": "opp_id"}), on=["competition_id", "season", "game_id"])
opponent = opponent[opponent.team_id != opponent.opp_id]
kp = keepers.merge(opponent, on=["competition_id", "season", "game_id", "team_id"]).merge(faced.rename(columns={"team_id": "opp_id"}), on=["competition_id", "season", "game_id", "opp_id"], how="left").fillna({"xg_on_target": 0, "goals": 0})
kp_season = kp.groupby(["competition_id", "season", "player_id"]).agg(minutes=("minutes", "sum"), matches=("game_id", "size"), xg_on_target=("xg_on_target", "sum"), goals=("goals", "sum")).reset_index()
kp_season = kp_season[kp_season.minutes >= quantities.MIN_MINUTES]
kp_season["prevented_per90"] = (kp_season.xg_on_target - kp_season.goals) / kp_season.minutes * 90
nxt = kp_season.assign(season=kp_season.season - 1)[["competition_id", "season", "player_id", "prevented_per90"]].rename(columns={"prevented_per90": "next"})
pairs = kp_season.merge(nxt, on=["competition_id", "season", "player_id"])
print(len(kp_season), "keeper-seasons ≥600 min | proxy mean", round(kp_season.prevented_per90.mean(), 3), "sd", round(kp_season.prevented_per90.std(), 3))
print(f"Understat proxy year-to-year r = {pairs.prevented_per90.corr(pairs.next):.3f} (n={len(pairs)}; Phase 0 on 3 PL seasons: 0.42–0.52)")
# FotMob goals prevented and Sofascore saves, for keepers with an identity
from scout.data import fotmob
fm = fotmob.load()
gp = fm[fm.stat == "_goals_prevented"][["competition_id", "season", "fotmob_player_id", "stat_value", "minutes_played"]].rename(columns={"stat_value": "goals_prevented"})
gp = gp[gp.minutes_played >= quantities.MIN_MINUTES]
gp_next = gp.assign(season=gp.season - 1)[["competition_id", "season", "fotmob_player_id", "goals_prevented"]].rename(columns={"goals_prevented": "next"})
gpp = gp.merge(gp_next, on=["competition_id", "season", "fotmob_player_id"])
print(f"FotMob goals prevented year-to-year r = {gpp.goals_prevented.corr(gpp.next):.3f} (n={len(gpp)}, all leagues on disk)")
gk_wr = wr[wr.role == "GK"]
print("Sofascore saves per 90 year-to-year r at 600 (Step 1):", wr_stability.loc[("GK", 600), "saves"])

1591 keeper-seasons ≥600 min | proxy mean -0.565 sd 0.248
Understat proxy year-to-year r = 0.336 (n=889; Phase 0 on 3 PL seasons: 0.42–0.52)


FotMob goals prevented year-to-year r = 0.362 (n=572, all leagues on disk)
Sofascore saves per 90 year-to-year r at 600 (Step 1): 0.36


Two follow-ups. Keepers: do the three measures agree, and is their average more stable than any
one? Outfield defensive *actions* (Step 1: r 0.6–0.7 within a club): do they travel with a mover,
which on/off did not?

In [16]:
# keepers: join the Understat proxy to FotMob goals prevented and Sofascore saves via Transfermarkt ids
fm_ids = identity.resolve_provider("fotmob", fm[fm.stat == "mins_played"].rename(columns={"stat_value": "fm_minutes"})[["competition_id", "season", "fotmob_player_id", "player_name", "team_name", "fm_minutes"]].drop_duplicates(["competition_id", "season", "fotmob_player_id"]), tm_side, lineage_all := build_team_lineage(tm_clubs, {"fotmob": fm[["competition_id", "team_name"]].drop_duplicates(), "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(), "understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"})}, load_overrides("teams")), people).drop_duplicates("provider_id")[["provider_id", "tm_player_id"]]
k = kp_season.copy(); k["tm_player_id"] = k.player_id.astype(int).astype(str).map(us_ids.set_index("provider_id").tm_player_id)
g = gp.copy(); g["tm_player_id"] = g.fotmob_player_id.astype(int).astype(str).map(fm_ids.set_index("provider_id").tm_player_id); g["gp_per90"] = g.goals_prevented / g.minutes_played * 90
sv = wr[wr.role == "GK"][["competition_id", "season", "tm_player_id", "saves", "goals_conceded"]]
three = k.merge(g[["competition_id", "season", "tm_player_id", "gp_per90"]], on=["competition_id", "season", "tm_player_id"]).merge(sv, on=["competition_id", "season", "tm_player_id"])
print(len(three), "keeper-seasons with all three measures")
print("correlations between measures:"); print(three[["prevented_per90", "gp_per90", "saves"]].corr().round(2).to_string())
z = (three[["prevented_per90", "gp_per90"]] - three[["prevented_per90", "gp_per90"]].mean()) / three[["prevented_per90", "gp_per90"]].std()
three["combined"] = z.mean(axis=1)
nxt = three.assign(season=three.season - 1)[["competition_id", "season", "tm_player_id", "prevented_per90", "gp_per90", "combined"]]
pairs = three.merge(nxt, on=["competition_id", "season", "tm_player_id"], suffixes=("", "_next"))
print("year-to-year r on the same keepers:", {c: round(pairs[c].corr(pairs[f"{c}_next"]), 3) for c in ["prevented_per90", "gp_per90", "combined"]}, "| n =", len(pairs))

486 keeper-seasons with all three measures
correlations between measures:
                 prevented_per90  gp_per90  saves
prevented_per90             1.00      0.45  -0.20
gp_per90                    0.45      1.00   0.01
saves                      -0.20      0.01   1.00
year-to-year r on the same keepers: {'prevented_per90': np.float64(0.42), 'gp_per90': np.float64(0.261), 'combined': np.float64(0.39)} | n = 245


In [17]:
# do defensive actions travel? same-role movers, per-90 actions at club A (season s) vs club B (s+1)
ss_clubs = ss[["competition_id", "season", "sofascore_player_id", "team_name"]].copy()
ss_clubs["club_id"] = ss_clubs.merge(lineage_all[lineage_all.provider == "sofascore"][["competition_id", "team_name", "club_id"]], on=["competition_id", "team_name"], how="left").club_id.values
wr_club = wr.merge(ss_clubs[["competition_id", "season", "sofascore_player_id", "club_id"]], on=["competition_id", "season", "sofascore_player_id"])
wr_club = wr_club[wr_club.minutes >= quantities.MIN_MINUTES].drop_duplicates(["season", "tm_player_id", "role"])
ACTIONS = ["tackles", "interceptions", "clearances", "recoveries", "accurate_passes", "dribbles"]
nxt = wr_club.assign(season=wr_club.season - 1)[["season", "tm_player_id", "role", "club_id"] + ACTIONS].rename(columns={"club_id": "club_next", **{a: f"{a}_next" for a in ACTIONS}})
pairs = wr_club.merge(nxt, on=["season", "tm_player_id", "role"])
rows = {}
for role in ["CB", "FB", "CM"]:
    for label, mask in [("stayers", pairs.club_id == pairs.club_next), ("movers", pairs.club_id != pairs.club_next)]:
        sub = pairs[(pairs.role == role) & mask]
        rows[(role, label)] = {a: round(sub[a].corr(sub[f"{a}_next"]), 2) for a in ACTIONS} | {"n": len(sub)}
print(pd.DataFrame(rows).T.to_string())

            tackles  interceptions  clearances  recoveries  accurate_passes  dribbles       n
CB stayers     0.59           0.64        0.67        0.65             0.84      0.56  1892.0
   movers      0.47           0.55        0.55        0.48             0.49      0.54   541.0
FB stayers     0.63           0.65        0.64        0.60             0.87      0.72  1562.0
   movers      0.53           0.64        0.54        0.39             0.53      0.69   432.0
CM stayers     0.66           0.65        0.69        0.55             0.87      0.74  1987.0
   movers      0.59           0.61        0.56        0.49             0.62      0.72   604.0


### Step 6 note — non-attacking roles and keepers, from the outputs above

**Defensive value cannot be measured from outcomes at this sample size — a negative result to
report, not to hide.** A real on/off (the team's non-penalty xG against per 90 in the matches a
player played versus the matches he missed, ≥ 5 of each, 17,327 player-club-seasons) has
year-to-year r between −0.03 and +0.08 in every role, for players who stay and for players who
move; its season-level noise (sd 0.32 xG per 90) swamps any player effect. Together with Step 2
(the proportional plus-minus measures the team), this kills the spec's "plus-minus is required
for non-attacking roles": in this data no on-pitch xG-difference construction carries a
player's defensive contribution. Rejected: both plus-minus forms.

**What does travel is what a defender does.** Tackles, interceptions, clearances and recoveries
per 90 keep r 0.47–0.64 after a move (0.59–0.69 for stayers) for CB, FB and CM — they are player
traits, not team artefacts. Decision: for every role the contribution core is expected output
(npxG + xA, Step 2); non-attacking roles additionally carry a **defensive activity profile**
(the four actions per 90, standardised within role) used for fit and similarity, and the
writeup says plainly that activity is measured while value is not. Ported:
`contribution.DEFENSIVE_ACTIONS`.

**Keepers stay in, on the Understat proxy alone.** On-target xG faced minus goals conceded per
90: year-to-year r 0.34 on 889 keeper-seasons (0.42 on the 245 that also have FotMob data —
2016-17 onward), above the 0.3 bar. FotMob's goals prevented is 0.26–0.36 and correlates 0.45
with the proxy; averaging the two (0.39) does not beat the proxy; Sofascore saves per 90 is a
volume count (r −0.20 with the proxy) and is not a quality measure. Rejected: FotMob GP as the
primary, the average, saves. The proxy's level is negative for everyone (mean −0.57 per 90)
because on-target xG is pre-shot; only the ranking is meaningful, and it gets the Step 9
shrinkage. Ported: `scout.models.keepers.prevented_per90`.

### Step 6 check — `scout.models.keepers` reproduces the proxy

In [18]:
from scout.models import keepers as keepers_model

shots_keyed = shots.assign(competition_id=shots.league.map(LEAGUE_TO_COMP))
packaged = keepers_model.prevented_per90(pm, shots_keyed, team_game)
packaged = packaged[packaged.minutes >= quantities.MIN_MINUTES]
nxt = packaged.assign(season=packaged.season - 1)[["competition_id", "season", "player_id", "prevented_per90"]].rename(columns={"prevented_per90": "next"})
pairs = packaged.merge(nxt, on=["competition_id", "season", "player_id"])
print(len(packaged), "keeper-seasons (above: 1,591) | mean", round(packaged.prevented_per90.mean(), 3), "(above: -0.565) | y2y r", round(pairs.prevented_per90.corr(pairs.next), 3), "(above: 0.336)")

1591 keeper-seasons (above: 1,591) | mean -0.565 (above: -0.565) | y2y r 0.336 (above: 0.336)


## Step 7 — League conversion factors (spec §4.B)

From movers: the same player with ≥ 600 minutes in league A in season s and in a different
league B in s+1. Understat exists only for the Big 5, so the cross-league measure is FotMob's
xG + xA per 90 on both sides (one provider, one definition; Step 5d showed it equals Sofascore's),
and Understat's npxG + xA is the cross-check for Big-5 ↔ Big-5 pairs. Factor = median log ratio
(after / before), with a bootstrap interval; age-controlled by residualising the log ratio on age
bands (the same player also ages a year). Pairs with fewer than 30 movers pool by tier.

In [19]:
players_tm = tm_loader.load_table("players")[["player_id", "date_of_birth"]].rename(columns={"player_id": "tm_player_id"})
players_tm["tm_player_id"] = players_tm.tm_player_id.astype(str)
fm_xg = fm[fm.stat.isin(["expected_goals", "expected_assists", "mins_played"])].pivot_table(index=["competition_id", "season", "fotmob_player_id"], columns="stat", values="stat_value", aggfunc="first").reset_index()
fm_xg = fm_xg.rename(columns={"mins_played": "minutes"}).dropna(subset=["minutes"])
fm_xg = fm_xg[fm_xg.minutes >= quantities.MIN_MINUTES].fillna({"expected_goals": 0.0, "expected_assists": 0.0})
fm_xg["output"] = (fm_xg.expected_goals + fm_xg.expected_assists) / fm_xg.minutes * 90
fm_xg["tm_player_id"] = fm_xg.fotmob_player_id.astype(int).astype(str).map(fm_ids.set_index("provider_id").tm_player_id)
fm_xg = fm_xg.dropna(subset=["tm_player_id"]).sort_values("minutes", ascending=False).drop_duplicates(["season", "tm_player_id"])
fm_xg = fm_xg.merge(players_tm, on="tm_player_id", how="left")
fm_xg["age"] = fm_xg.season + 1 - pd.to_datetime(fm_xg.date_of_birth).dt.year  # age in the season's spring
print(len(fm_xg), "FotMob player-seasons ≥600 min with a Transfermarkt id |", fm_xg.groupby("competition_id").season.nunique().to_dict())
nxt = fm_xg.assign(season=fm_xg.season - 1)[["season", "tm_player_id", "competition_id", "output"]].rename(columns={"competition_id": "league_to", "output": "output_after"})
moves = fm_xg.merge(nxt, on=["season", "tm_player_id"])
moves = moves[moves.competition_id != moves.league_to].copy()
moves["log_ratio"] = np.log((moves.output_after + 0.02) / (moves.output + 0.02))  # +0.02 keeps zero-output seasons finite
print(len(moves), "cross-league movers |", moves.groupby(["competition_id", "league_to"]).size().sort_values(ascending=False).head(12).to_dict())

32518 FotMob player-seasons ≥600 min with a Transfermarkt id | {'A1': 8, 'BE1': 8, 'BRA1': 10, 'C1': 8, 'DK1': 9, 'ES1': 10, 'FR1': 10, 'GB1': 10, 'IT1': 10, 'NL1': 9, 'PO1': 9, 'TR1': 8}
2374 cross-league movers | {('FR1', 'GB1'): 97, ('ES1', 'GB1'): 86, ('FR1', 'IT1'): 86, ('IT1', 'GB1'): 72, ('GB1', 'ES1'): 72, ('GB1', 'IT1'): 71, ('FR1', 'ES1'): 61, ('ES1', 'IT1'): 58, ('PO1', 'TR1'): 57, ('FR1', 'TR1'): 54, ('IT1', 'FR1'): 54, ('IT1', 'TR1'): 50}


In [20]:
BIG5 = set(config.BIG5)
moves["tier_from"] = np.where(moves.competition_id.isin(BIG5), "big5", "feeder")
moves["tier_to"] = np.where(moves.league_to.isin(BIG5), "big5", "feeder")
# age control: residualise the log ratio on age bands across all movers
moves["age_band"] = pd.cut(moves.age, [15, 21, 24, 27, 30, 45], labels=["≤21", "22-24", "25-27", "28-30", "31+"])
age_effect = moves.groupby("age_band", observed=True).log_ratio.mean()
print("mean log ratio by age band (all movers):", age_effect.round(3).to_dict())
moves["adj"] = moves.log_ratio - moves.age_band.map(age_effect).astype(float) + moves.log_ratio.mean()


def factor(frame, col="adj", n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    x = frame[col].to_numpy()
    med = np.median(x)
    boots = [np.median(rng.choice(x, len(x))) for _ in range(n_boot)]
    return round(float(np.exp(med)), 3), round(float(np.exp(np.percentile(boots, 10))), 3), round(float(np.exp(np.percentile(boots, 90))), 3)


rows = []
for (a, b), g in moves.groupby(["competition_id", "league_to"]):
    if len(g) >= 30:
        f, lo, hi = factor(g); rows.append((a, b, len(g), f, lo, hi))
pairs_table = pd.DataFrame(rows, columns=["from", "to", "n", "factor", "p10", "p90"]).sort_values("n", ascending=False)
print("league pairs with ≥30 movers (factor = multiplier on xG+xA per 90 after the move, age-adjusted, 80% bootstrap interval):")
print(pairs_table.to_string(index=False))
print("\npooled by tier:")
for (a, b), g in moves.groupby(["tier_from", "tier_to"]):
    print(f"  {a} → {b}: n={len(g)}, factor {factor(g)}")

mean log ratio by age band (all movers): {'≤21': 0.263, '22-24': 0.135, '25-27': 0.176, '28-30': 0.053, '31+': 0.197}


league pairs with ≥30 movers (factor = multiplier on xG+xA per 90 after the move, age-adjusted, 80% bootstrap interval):
from  to  n  factor   p10   p90
 FR1 GB1 97   0.961 0.899 0.979
 FR1 IT1 86   0.979 0.914 1.011
 ES1 GB1 86   1.020 0.979 1.032
 GB1 ES1 72   1.057 1.020 1.095
 IT1 GB1 72   0.979 0.929 0.979
 GB1 IT1 71   1.020 0.979 1.107
 FR1 ES1 61   0.979 0.979 0.979
 ES1 IT1 58   0.979 0.979 1.020
 PO1 TR1 57   1.020 0.987 1.097
 FR1 TR1 54   0.979 0.968 1.020
 IT1 FR1 54   1.020 0.979 1.020
 IT1 TR1 50   1.044 0.979 1.153
BRA1 PO1 49   1.107 1.020 1.221
 IT1 ES1 48   0.999 0.968 1.096
 ES1 FR1 47   1.049 0.979 1.107
 GB1 FR1 47   1.107 1.107 1.197
 BE1 FR1 47   0.858 0.782 0.920
 GB1 TR1 39   1.076 0.979 1.137
 BE1 TR1 38   0.979 0.958 0.984
 NL1 BE1 35   1.020 0.979 1.163
 NL1 GB1 34   0.563 0.459 0.706
 PO1 FR1 33   1.020 0.979 1.030
 FR1 BE1 32   1.020 0.979 1.112
 BE1 IT1 32   0.648 0.602 0.827
 NL1 IT1 30   0.897 0.686 0.957

pooled by tier:
  big5 → big5: n=799, factor (

In [21]:
# Kill check: do held-out movers land inside the interval? Leave-one-season-out on the tier pools.
coverage = []
for (a, b), g in moves.groupby(["tier_from", "tier_to"]):
    for s in sorted(g.season.unique()):
        train, test = g[g.season != s], g[g.season == s]
        if len(train) < 30 or len(test) < 5:
            continue
        f, lo, hi = factor(train)
        # prediction interval for an individual mover: the factor ± the spread of individual ratios (10th-90th pct of adj residuals)
        spread = np.percentile(train.adj - np.median(train.adj), [10, 90])
        inside = ((test.adj >= np.log(f) + spread[0]) & (test.adj <= np.log(f) + spread[1])).mean()
        coverage.append((a, b, s, len(test), round(inside, 2)))
cov = pd.DataFrame(coverage, columns=["from", "to", "season", "n_test", "inside_80pct"])
print("held-out season coverage of the 80% individual interval, by tier pair:")
print(cov.groupby(["from", "to"]).apply(lambda d: pd.Series({"seasons": len(d), "movers": d.n_test.sum(), "coverage": round(np.average(d.inside_80pct, weights=d.n_test), 3)})).to_string())
# Big-5 ↔ Big-5 cross-check with Understat npxG + xA (different provider, penalty-free)
us_out = per90[per90.minutes >= quantities.MIN_MINUTES].copy(); us_out["output"] = us_out.npxg + us_out.xa
us_out = us_out.sort_values("minutes", ascending=False).drop_duplicates(["season", "player_id"])
nxt = us_out.assign(season=us_out.season - 1)[["season", "player_id", "competition_id", "output"]].rename(columns={"competition_id": "league_to", "output": "output_after"})
us_moves = us_out.merge(nxt, on=["season", "player_id"]); us_moves = us_moves[us_moves.competition_id != us_moves.league_to]
us_moves["log_ratio"] = np.log((us_moves.output_after + 0.02) / (us_moves.output + 0.02))
print(f"\nUnderstat Big-5 ↔ Big-5 movers: n={len(us_moves)}, factor {factor(us_moves, 'log_ratio')} | FotMob on the same tier pair: {factor(moves[(moves.tier_from == 'big5') & (moves.tier_to == 'big5')], 'log_ratio')}")

held-out season coverage of the 80% individual interval, by tier pair:
               seasons  movers  coverage
from   to                               
big5   big5        9.0   799.0     0.796
       feeder      9.0   399.0     0.794
feeder big5        8.0   521.0     0.743
       feeder      8.0   654.0     0.772

Understat Big-5 ↔ Big-5 movers: n=1259, factor (0.988, 0.968, 1.0) | FotMob on the same tier pair: (1.0, 1.0, 1.0)


The medians above sit on spikes (0.979 / 1.020 repeated; FR1 → ES1 has p10 = p90): many movers
have an identical ratio, which points at FotMob's `expected_goals` *total* list being absent for
players and filled as 0 (the OPEN absence-semantics item). Check presence, then redo the factors on
FotMob's per-90 xG + xA list (NaN when absent, never 0), only for movers with real output before
the move, and compare the ratio form with a difference form.

In [22]:
fm_raw = fm[fm.stat.isin(["expected_goals", "_expected_goals_and_expected_assists_per_90", "mins_played"])].pivot_table(index=["competition_id", "season", "fotmob_player_id"], columns="stat", values="stat_value", aggfunc="first").reset_index()
fm_raw = fm_raw[fm_raw.mins_played >= quantities.MIN_MINUTES]
print("FotMob player-seasons ≥600 min:", len(fm_raw), "| expected_goals present:", f"{fm_raw.expected_goals.notna().mean():.1%}", "| xG+xA per-90 list present:", f"{fm_raw._expected_goals_and_expected_assists_per_90.notna().mean():.1%}")
print("expected_goals present by league:", fm_raw.groupby("competition_id").expected_goals.apply(lambda c: round(c.notna().mean(), 2)).to_dict())
print("share of movers above with output exactly equal before and after:", f"{(moves.output == moves.output_after).mean():.1%}", "| with output 0 before:", f"{(moves.output == 0).mean():.1%}")

FotMob player-seasons ≥600 min: 35768 | expected_goals present: 60.3% | xG+xA per-90 list present: 54.1%
expected_goals present by league: {'A1': 0.71, 'BE1': 0.73, 'BRA1': 0.52, 'C1': 0.74, 'DK1': 0.65, 'ES1': 0.57, 'FR1': 0.57, 'GB1': 0.58, 'IT1': 0.57, 'NL1': 0.64, 'PO1': 0.62, 'TR1': 0.52}
share of movers above with output exactly equal before and after: 26.4% | with output 0 before: 38.0%


In [23]:
# Redo on the per-90 list; movers with ≥ 0.10 xG+xA per 90 before the move (a real attacking output to convert)
fm90 = fm_raw.rename(columns={"_expected_goals_and_expected_assists_per_90": "output", "mins_played": "minutes"}).dropna(subset=["output"])
fm90["tm_player_id"] = fm90.fotmob_player_id.astype(int).astype(str).map(fm_ids.set_index("provider_id").tm_player_id)
fm90 = fm90.dropna(subset=["tm_player_id"]).sort_values("minutes", ascending=False).drop_duplicates(["season", "tm_player_id"]).merge(players_tm, on="tm_player_id", how="left")
fm90["age"] = fm90.season + 1 - pd.to_datetime(fm90.date_of_birth).dt.year
nxt = fm90.assign(season=fm90.season - 1)[["season", "tm_player_id", "competition_id", "output"]].rename(columns={"competition_id": "league_to", "output": "output_after"})
mv = fm90.merge(nxt, on=["season", "tm_player_id"]); mv = mv[(mv.competition_id != mv.league_to) & (mv.output >= 0.10)].copy()
mv["log_ratio"] = np.log(mv.output_after.clip(lower=0.02) / mv.output)
mv["diff"] = mv.output_after - mv.output
mv["tier_from"] = np.where(mv.competition_id.isin(BIG5), "big5", "feeder"); mv["tier_to"] = np.where(mv.league_to.isin(BIG5), "big5", "feeder")
mv["age_band"] = pd.cut(mv.age, [15, 21, 24, 27, 30, 45], labels=["≤21", "22-24", "25-27", "28-30", "31+"])
for col in ["log_ratio", "diff"]:
    eff = mv.groupby("age_band", observed=True)[col].mean()
    mv[f"{col}_adj"] = mv[col] - mv.age_band.map(eff).astype(float) + mv[col].mean()
print(len(mv), "movers with ≥0.10 per 90 before | age effect on log ratio:", mv.groupby("age_band", observed=True).log_ratio.mean().round(3).to_dict())
print("\nleague pairs with ≥30 movers — ratio form (age-adjusted, 80% bootstrap):")
rows = [(a, b, len(g), *factor(g, "log_ratio_adj")) for (a, b), g in mv.groupby(["competition_id", "league_to"]) if len(g) >= 30]
print(pd.DataFrame(rows, columns=["from", "to", "n", "factor", "p10", "p90"]).sort_values("n", ascending=False).to_string(index=False))
print("\npooled by tier — ratio form | difference form (per 90):")
for (a, b), g in mv.groupby(["tier_from", "tier_to"]):
    d = g.diff_adj.to_numpy(); rng = np.random.default_rng(0); boots = [np.median(rng.choice(d, len(d))) for _ in range(500)]
    print(f"  {a} → {b}: n={len(g)} | ratio {factor(g, 'log_ratio_adj')} | diff {round(float(np.median(d)), 3)} [{round(float(np.percentile(boots, 10)), 3)}, {round(float(np.percentile(boots, 90)), 3)}]")

718 movers with ≥0.10 per 90 before | age effect on log ratio: {'≤21': -0.066, '22-24': -0.246, '25-27': -0.18, '28-30': -0.25, '31+': -0.224}

league pairs with ≥30 movers — ratio form (age-adjusted, 80% bootstrap):
from  to  n  factor   p10   p90
 FR1 GB1 35   0.672 0.606 0.734

pooled by tier — ratio form | difference form (per 90):
  big5 → big5: n=224 | ratio (0.849, 0.809, 0.885) | diff -0.035 [-0.045, -0.023]
  big5 → feeder: n=102 | ratio (1.211, 1.178, 1.305) | diff 0.069 [0.039, 0.083]
  feeder → big5: n=195 | ratio (0.669, 0.631, 0.711) | diff -0.093 [-0.121, -0.071]
  feeder → feeder: n=197 | ratio (0.902, 0.872, 0.941) | diff -0.023 [-0.031, -0.011]


In [24]:
# Held-out coverage and error, ratio vs difference, by tier pair (leave-one-season-out)
def holdout(frame, form):
    col = "log_ratio_adj" if form == "ratio" else "diff_adj"
    out = []
    for s in sorted(frame.season.unique()):
        train, test = frame[frame.season != s], frame[frame.season == s]
        if len(train) < 30 or len(test) < 5:
            continue
        centre = np.median(train[col]); spread = np.percentile(train[col] - centre, [10, 90])
        if form == "ratio":
            pred = test.output * np.exp(centre); lo = test.output * np.exp(centre + spread[0]); hi = test.output * np.exp(centre + spread[1])
        else:
            pred = test.output + centre; lo = test.output + centre + spread[0]; hi = test.output + centre + spread[1]
        out.append((len(test), ((test.output_after >= lo) & (test.output_after <= hi)).mean(), np.abs(test.output_after - pred).mean()))
    n = sum(o[0] for o in out)
    return {"movers": n, "coverage_80": round(sum(o[0] * o[1] for o in out) / n, 3), "mae": round(sum(o[0] * o[2] for o in out) / n, 4)}


print("held-out, by tier pair:")
for (a, b), g in mv.groupby(["tier_from", "tier_to"]):
    print(f"  {a} → {b}: ratio {holdout(g, 'ratio')} | diff {holdout(g, 'diff')} | no-factor baseline mae {round(float(np.abs(g.output_after - g.output).mean()), 4)}")

held-out, by tier pair:
  big5 → big5: ratio {'movers': 223, 'coverage_80': np.float64(0.794), 'mae': np.float64(0.0975)} | diff {'movers': 223, 'coverage_80': np.float64(0.839), 'mae': np.float64(0.1028)} | no-factor baseline mae 0.11
  big5 → feeder: ratio {'movers': 102, 'coverage_80': np.float64(0.804), 'mae': np.float64(0.1217)} | diff {'movers': 102, 'coverage_80': np.float64(0.794), 'mae': np.float64(0.1267)} | no-factor baseline mae 0.1408
  feeder → big5: ratio {'movers': 195, 'coverage_80': np.float64(0.774), 'mae': np.float64(0.0876)} | diff {'movers': 195, 'coverage_80': np.float64(0.795), 'mae': np.float64(0.1213)} | no-factor baseline mae 0.1488
  feeder → feeder: ratio {'movers': 197, 'coverage_80': np.float64(0.782), 'mae': np.float64(0.1052)} | diff {'movers': 197, 'coverage_80': np.float64(0.787), 'mae': np.float64(0.1104)} | no-factor baseline mae 0.111


### Step 7 note — league conversion factors, from the outputs above

**Two findings before the factors.** (1) FotMob's *total* lists (xG, xA, …) are absent for 40%
of player-seasons with ≥ 600 minutes; absence is not zero — 38% of the first mover set had
"output 0 before" and 26% an identical output on both sides, which put the medians on spikes.
Decision on the OPEN item: a missing FotMob list value is NaN, never 0, for every stat.
(2) The per-90 xG + xA list (54% present, NaN when absent) with a ≥ 0.10 output-before floor
gives 718 clean movers, all leagues.

**Chosen: the ratio form, pooled by tier, with league pairs where ≥ 30 movers exist.** Factors
are age-adjusted median multipliers on xG + xA per 90 (80% bootstrap interval on the median; the
individual interval is the 10th–90th percentile of movers' log ratios):

| pair | movers | factor | held-out 80% coverage | held-out MAE (no factor) |
|---|---|---|---|---|
| feeder → Big 5 | 195 | **0.67** [0.63, 0.71] | 0.77 | 0.088 (0.149) |
| Big 5 → feeder | 102 | 1.21 [1.18, 1.31] | 0.80 | 0.122 (0.141) |
| Big 5 → Big 5 | 224 | 0.85 [0.81, 0.89] | 0.79 | 0.098 (0.110) |
| feeder → feeder | 197 | 0.90 [0.87, 0.94] | 0.78 | 0.105 (0.111) |
| FR1 → GB1 (only pair ≥ 30) | 35 | 0.67 [0.61, 0.73] | – | – |

- The Big-5 → Big-5 factor of 0.85 is not a league effect: it is regression to the mean —
  players move after a good season. Every pair carries it, so the league-specific part of
  feeder → Big 5 is about 0.67 / 0.85 ≈ 0.79; the model applies the raw factor because a
  candidate scouted on a good feeder season is exactly a past mover.
- Ratio beats difference on held-out error in every tier (feeder → Big 5: 0.088 vs 0.121), and
  both beat "no factor". Coverage 0.77–0.80 against 0.80 nominal passes the kill check; the
  feeder → Big 5 interval is widened by its shortfall in Step 9. Rejected: the difference form;
  per-league-pair factors below 30 movers (one pair qualifies today).
- The age adjustment is negative for every band above 21 (−0.18 to −0.25 log): older movers
  lose more output, ≤ 21 movers almost none — Phase 3's trajectory model owns that.

Ported: `scout.models.league_factors` (`log_ratios`, `factor`, `individual_spread`, `tier_factors`).

### Step 7 check — `scout.models.league_factors` reproduces the tier table

In [25]:
from scout.models import league_factors

table = league_factors.tier_factors(mv[["competition_id", "league_to", "output", "output_after", "age"]], set(config.BIG5))
print(table.round(3).to_string(index=False))
print("(above: feeder→big5 0.669 [0.631, 0.711] n=195; big5→feeder 1.211; big5→big5 0.849; feeder→feeder 0.902; FR1→GB1 0.672 n=35)")

  from     to  movers  factor   p10   p90  spread_lo  spread_hi
  big5   big5     224   0.849 0.809 0.885     -0.589      0.614
  big5 feeder     102   1.211 1.178 1.305     -0.858      0.546
feeder   big5     195   0.669 0.631 0.711     -0.572      0.470
feeder feeder     197   0.902 0.872 0.941     -0.708      0.543
   FR1    GB1      35   0.672 0.606 0.734     -0.560      0.441
(above: feeder→big5 0.669 [0.631, 0.711] n=195; big5→feeder 1.211; big5→big5 0.849; feeder→feeder 0.902; FR1→GB1 0.672 n=35)
